# 0.3 RPC BAL RLP Estimation

This notebook estimates raw RLP BAL bytes using JSON-RPC `debug_traceBlockByNumber` with `prestateTracer`, following Toni's `eth-bal-analysis` builder logic.

It does **not** compute calldata from RPC. It reads calldata bytes from the CSV produced by `0.2-calldata-xatu.ipynb`, writes a separate RPC BAL summary CSV, and merges BAL bytes back into the calldata CSV.

## Bandwidth Join

```text
bandwidth_rlp_bytes = xatu_calldata_bytes + rpc_bal_rlp_bytes
```

Each BAL account entry is encoded as:

```text
[address, storage_writes, storage_reads, balance_changes, nonce_changes, code_changes]
```

In [6]:
import importlib
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

import sys
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import sim.rpc_bal as rpc_bal
rpc_bal = importlib.reload(rpc_bal)

BAL_SEMANTICS = rpc_bal.BAL_SEMANTICS
build_rpc_bal_for_block = rpc_bal.build_rpc_bal_for_block

load_dotenv(PROJECT_ROOT / ".env")
RPC_URL = os.environ.get("ALCHEMY_RPC")
if not RPC_URL:
    raise RuntimeError("Missing ALCHEMY_RPC in .env")

# Do not print RPC_URL; it contains the API key.
print("Loaded ALCHEMY_RPC")

Loaded ALCHEMY_RPC


In [7]:
START_BLOCK = 22_886_891
N_BLOCKS = 50
BLOCKS = list(range(START_BLOCK, START_BLOCK + N_BLOCKS))
CALLDATA_CSV = PROJECT_ROOT / "data" / f"xatu_calldata_{min(BLOCKS)}_{max(BLOCKS)}.csv"
SUMMARY_CSV = PROJECT_ROOT / "data" / f"rpc_bal_summary_{min(BLOCKS)}_{max(BLOCKS)}.csv"

INCLUDE_READS = True

# True estimates the fuller EIP-7928 block-level BAL payload.
INCLUDE_SYSTEM_CHANGES = True
BAL_SEMANTICS_VERSION = BAL_SEMANTICS

WRITE_CSV = True
WRITE_RLP = False

In [8]:
if not CALLDATA_CSV.exists():
    raise FileNotFoundError(
        f"Missing calldata CSV: {CALLDATA_CSV}. Run notebooks/0.2-calldata-xatu.ipynb first."
    )

calldata = pd.read_csv(CALLDATA_CSV)
missing_blocks = sorted(set(BLOCKS) - set(calldata["block_number"].astype(int)))
if missing_blocks:
    raise RuntimeError(f"Calldata CSV is missing blocks: {missing_blocks}")

calldata_by_block = calldata.set_index("block_number")["calldata_bytes"].astype(int).to_dict()
display(calldata)

,block_number,slot,n_txs_from_payload,n_txs,calldata_bytes,bal_bytes,bandwidth_bytes,execution_tx_rows,execution_calldata_bytes,calldata_zero_bytes,...,balance_changes,nonce_changes,code_changes,code_bytes,storage_writes_rlp_bytes,storage_reads_rlp_bytes,balance_changes_rlp_bytes,nonce_changes_rlp_bytes,code_changes_rlp_bytes,account_shell_rlp_bytes
0,22886891,12108642,268,268,63197,120499,183696,268,63197,39716,...,708,271,2,16479,52323,26754,8836,1520,16501,14565
1,22886892,12108643,183,183,76877,104914,181791,183,76877,54798,...,494,183,0,0,46354,36975,5874,1006,0,14705
2,22886893,12108644,162,162,78355,88311,166666,162,78355,55493,...,473,162,0,0,41806,26447,5763,877,0,13418
3,22886894,12108645,122,122,40638,67500,108138,122,40638,27441,...,337,123,2,46,27422,25112,3859,654,54,10399
4,22886895,12108646,223,223,56590,112929,169519,223,56590,39140,...,594,223,0,0,48862,37678,7267,1255,0,17867
5,22886896,12108647,282,282,65928,108694,174622,282,65928,46763,...,700,282,0,0,50794,31903,8666,1545,0,15786
6,22886897,12108648,155,155,35659,66531,102190,155,35659,21937,...,402,156,1,23,32915,17901,4640,776,27,10272
7,22886898,12108649,238,238,78196,104874,183070,238,78196,51743,...,640,241,1,45,45687,34251,7703,1348,49,15836
8,22886899,12108650,165,165,43392,69183,112575,165,43392,26726,...,454,166,1,23,31868,20168,5276,863,27,10981
9,22886900,12108651,263,263,54982,101106,156088,263,54982,37451,...,727,265,1,573,45294,29212,8893,1409,583,15715


In [9]:
summary_cols = [
    "block_number",
    "bal_semantics",
    "include_reads",
    "include_system_changes",
    "calldata_source",
    "calldata_bytes",
    "bal_rlp_bytes",
    "bandwidth_rlp_bytes",
    "accounts",
    "storage_write_slots",
    "storage_write_changes",
    "storage_reads",
    "balance_changes",
    "nonce_changes",
    "code_changes",
    "code_bytes",
    "storage_writes_rlp_bytes",
    "storage_reads_rlp_bytes",
    "balance_changes_rlp_bytes",
    "nonce_changes_rlp_bytes",
    "code_changes_rlp_bytes",
    "account_shell_rlp_bytes",
]

rows = []
if SUMMARY_CSV.exists():
    prior = pd.read_csv(SUMMARY_CSV)
    required_cache_cols = {"bal_semantics", "include_reads", "include_system_changes"}
    if required_cache_cols.issubset(prior.columns):
        prior = prior[
            (prior["bal_semantics"] == BAL_SEMANTICS_VERSION)
            & (prior["include_reads"].astype(bool) == INCLUDE_READS)
            & (prior["include_system_changes"].astype(bool) == INCLUDE_SYSTEM_CHANGES)
        ]
    else:
        prior = prior.iloc[0:0]
    available_summary_cols = [col for col in summary_cols if col in prior.columns]
    rows.extend(prior[available_summary_cols].to_dict("records"))

seen = {int(row["block_number"]) for row in rows}
rlp_outputs = {}

for block_number in BLOCKS:
    if int(block_number) in seen:
        print(f"Skipping block {block_number}; already in {SUMMARY_CSV.name}")
        continue
    print(f"Building RPC BAL for block {block_number}...")
    result = build_rpc_bal_for_block(
        RPC_URL,
        block_number,
        calldata_bytes=calldata_by_block[block_number],
        include_reads=INCLUDE_READS,
        include_system_changes=INCLUDE_SYSTEM_CHANGES,
    )
    rows.append(result.summary.as_dict())
    seen.add(int(block_number))
    rlp_outputs[block_number] = result.rlp_bytes
    if WRITE_CSV:
        data_dir = PROJECT_ROOT / "data"
        data_dir.mkdir(exist_ok=True)
        pd.DataFrame(rows).reindex(columns=summary_cols).drop_duplicates("block_number", keep="last").sort_values("block_number").to_csv(SUMMARY_CSV, index=False)

summary = pd.DataFrame(rows).reindex(columns=summary_cols).drop_duplicates("block_number", keep="last").sort_values("block_number")
display(summary)

if WRITE_CSV:
    data_dir = PROJECT_ROOT / "data"
    data_dir.mkdir(exist_ok=True)
    summary.to_csv(SUMMARY_CSV, index=False)
    print(SUMMARY_CSV)

merge_cols = [col for col in summary_cols if col not in {"calldata_bytes", "calldata_source"}]
stale_cols = [col for col in merge_cols if col != "block_number" and col in calldata.columns]
merged = calldata.drop(columns=stale_cols + [col for col in ["bal_bytes", "bandwidth_bytes"] if col in calldata.columns])
merged = merged.merge(summary[merge_cols], on="block_number", how="left", validate="one_to_one")
merged["bal_bytes"] = merged["bal_rlp_bytes"].astype("Int64")
merged["bandwidth_bytes"] = (merged["calldata_bytes"] + merged["bal_bytes"]).astype("Int64")

front = []
for col in merged.columns:
    if col in {"bal_bytes", "bandwidth_bytes"}:
        continue
    front.append(col)
    if col == "calldata_bytes":
        front.extend(["bal_bytes", "bandwidth_bytes"])
merged = merged[front + [col for col in merged.columns if col not in front]]
display(merged)

if WRITE_CSV:
    merged.to_csv(CALLDATA_CSV, index=False)
    print(CALLDATA_CSV)

if WRITE_RLP:
    suffix = "with_reads" if INCLUDE_READS else "without_reads"
    for block_number, payload in rlp_outputs.items():
        data_dir = PROJECT_ROOT / "data"
        data_dir.mkdir(exist_ok=True)
        out = data_dir / f"rpc_bal_{block_number}_{suffix}.rlp"
        out.write_bytes(payload)
        print(out)

Skipping block 22886891; already in rpc_bal_summary_22886891_22886940.csv
Skipping block 22886892; already in rpc_bal_summary_22886891_22886940.csv
Skipping block 22886893; already in rpc_bal_summary_22886891_22886940.csv
Skipping block 22886894; already in rpc_bal_summary_22886891_22886940.csv
Skipping block 22886895; already in rpc_bal_summary_22886891_22886940.csv
Skipping block 22886896; already in rpc_bal_summary_22886891_22886940.csv
Skipping block 22886897; already in rpc_bal_summary_22886891_22886940.csv
Skipping block 22886898; already in rpc_bal_summary_22886891_22886940.csv
Skipping block 22886899; already in rpc_bal_summary_22886891_22886940.csv
Skipping block 22886900; already in rpc_bal_summary_22886891_22886940.csv
Skipping block 22886901; already in rpc_bal_summary_22886891_22886940.csv
Skipping block 22886902; already in rpc_bal_summary_22886891_22886940.csv
Skipping block 22886903; already in rpc_bal_summary_22886891_22886940.csv
Skipping block 22886904; already in rp

,block_number,bal_semantics,include_reads,include_system_changes,calldata_source,calldata_bytes,bal_rlp_bytes,bandwidth_rlp_bytes,accounts,storage_write_slots,...,balance_changes,nonce_changes,code_changes,code_bytes,storage_writes_rlp_bytes,storage_reads_rlp_bytes,balance_changes_rlp_bytes,nonce_changes_rlp_bytes,code_changes_rlp_bytes,account_shell_rlp_bytes
0,22886891,eip7928_pre_tx_post_indices_v1,True,True,xatu,63197,120499,183696,559,611,...,708,271,2,16479,52323,26754,8836,1520,16501,14565
1,22886892,eip7928_pre_tx_post_indices_v1,True,True,xatu,76877,104914,181791,559,602,...,494,183,0,0,46354,36975,5874,1006,0,14705
2,22886893,eip7928_pre_tx_post_indices_v1,True,True,xatu,78355,88311,166666,511,535,...,473,162,0,0,41806,26447,5763,877,0,13418
3,22886894,eip7928_pre_tx_post_indices_v1,True,True,xatu,40638,67500,108138,396,373,...,337,123,2,46,27422,25112,3859,654,54,10399
4,22886895,eip7928_pre_tx_post_indices_v1,True,True,xatu,56590,112929,169519,680,665,...,594,223,0,0,48862,37678,7267,1255,0,17867
5,22886896,eip7928_pre_tx_post_indices_v1,True,True,xatu,65928,108694,174622,606,644,...,700,282,0,0,50794,31903,8666,1545,0,15786
6,22886897,eip7928_pre_tx_post_indices_v1,True,True,xatu,35659,66531,102190,394,399,...,402,156,1,23,32915,17901,4640,776,27,10272
7,22886898,eip7928_pre_tx_post_indices_v1,True,True,xatu,78196,104874,183070,605,611,...,640,241,1,45,45687,34251,7703,1348,49,15836
8,22886899,eip7928_pre_tx_post_indices_v1,True,True,xatu,43392,69183,112575,420,406,...,454,166,1,23,31868,20168,5276,863,27,10981
9,22886900,eip7928_pre_tx_post_indices_v1,True,True,xatu,54982,101106,156088,603,593,...,727,265,1,573,45294,29212,8893,1409,583,15715


/Users/william/PycharmProjects/eip-7999-research/data/rpc_bal_summary_22886891_22886940.csv


,block_number,slot,n_txs_from_payload,n_txs,calldata_bytes,bal_bytes,bandwidth_bytes,execution_tx_rows,execution_calldata_bytes,calldata_zero_bytes,...,balance_changes,nonce_changes,code_changes,code_bytes,storage_writes_rlp_bytes,storage_reads_rlp_bytes,balance_changes_rlp_bytes,nonce_changes_rlp_bytes,code_changes_rlp_bytes,account_shell_rlp_bytes
0,22886891,12108642,268,268,63197,120499,183696,268,63197,39716,...,708,271,2,16479,52323,26754,8836,1520,16501,14565
1,22886892,12108643,183,183,76877,104914,181791,183,76877,54798,...,494,183,0,0,46354,36975,5874,1006,0,14705
2,22886893,12108644,162,162,78355,88311,166666,162,78355,55493,...,473,162,0,0,41806,26447,5763,877,0,13418
3,22886894,12108645,122,122,40638,67500,108138,122,40638,27441,...,337,123,2,46,27422,25112,3859,654,54,10399
4,22886895,12108646,223,223,56590,112929,169519,223,56590,39140,...,594,223,0,0,48862,37678,7267,1255,0,17867
5,22886896,12108647,282,282,65928,108694,174622,282,65928,46763,...,700,282,0,0,50794,31903,8666,1545,0,15786
6,22886897,12108648,155,155,35659,66531,102190,155,35659,21937,...,402,156,1,23,32915,17901,4640,776,27,10272
7,22886898,12108649,238,238,78196,104874,183070,238,78196,51743,...,640,241,1,45,45687,34251,7703,1348,49,15836
8,22886899,12108650,165,165,43392,69183,112575,165,43392,26726,...,454,166,1,23,31868,20168,5276,863,27,10981
9,22886900,12108651,263,263,54982,101106,156088,263,54982,37451,...,727,265,1,573,45294,29212,8893,1409,583,15715


/Users/william/PycharmProjects/eip-7999-research/data/xatu_calldata_22886891_22886940.csv


In [10]:
# Optional local calibration against nerolation/eth-bal-analysis raw RLP samples.
sample_dir = Path("/private/tmp/eth-bal-analysis/bal_raw/rlp")
calibration_rows = []
if sample_dir.exists():
    suffix = "with_reads" if INCLUDE_READS else "without_reads"
    for block_number in BLOCKS:
        sample = sample_dir / f"{block_number}_{suffix}.rlp"
        if sample.exists():
            sample_bytes = sample.stat().st_size
            row = summary[summary["block_number"] == block_number].iloc[0]
            calibration_rows.append({
                "block_number": block_number,
                "rpc_bal_rlp_bytes": int(row["bal_rlp_bytes"]),
                "sample_bal_rlp_bytes": sample_bytes,
                "delta_bytes": int(row["bal_rlp_bytes"]) - sample_bytes,
            })

calibration = pd.DataFrame(calibration_rows)
display(calibration)

,block_number,rpc_bal_rlp_bytes,sample_bal_rlp_bytes,delta_bytes
0,22886891,120499,119857,642
1,22886892,104914,104376,538
2,22886893,88311,87790,521
